**Your name:** Kyle Whitecross

**Your student ID number:** 34736671

**Shared link to this notebook:**

# [COMPSCI 446: Search Engines - Spring 2025 ](https://people.cs.umass.edu/~rahimi/teaching/spring2025/cs446.html)
# Programming Assignment 3 (P3): Indexing & Retrieval Models (Total : 100 points)



**Description**

This project is focused on indexing and document retrieval on a small collection of documents. This assignment is related to Chapter 5 in the textbook, especially section 5.3, which covers inverted indexes, and section 5.6 (with an emphasis on 5.6.1) which discusses index construction. For retrieval models, section 5.7.2 as well as sections 7.1.2, 7.2.2, and 7.3.1, will be particularly useful.



**Instructions**

* To start working on this homework, you would first need to save the notebook to your local Google Drive. For this purpose, you can click on *Copy to Drive* button on the upper left (or *Save a copy in Drive* under *File*). You can alternatively click the *Share* button located at the top right corner and click on *Copy Link* under *Get Link* to get a link and copy this notebook to your Google Drive.  

* Next, open the copy in your Google Drive and close the original. This way, you'll be working on your own version and won't accidentally waste time on a copy that can't be saved.

*   You can download this notebook (file ipynb) and run it on your local machine, then upload the result file to Google Colab.

* The following instructions assume you are working in your own copy of the notebook.

*   For questions with descriptive answers, please replace the text in the cell which states "Enter your answer here!" with your answer. If you are using mathematical notation in your answers, please define the variables.

*   For coding questions, you can add code where it says "enter code here" and execute the cell to get the output/results.

**Submission Details**

* Due date: **TBD**

* When you are done, you will upload the result for grading.

* Before starting the interesting part of this project, click the *Share* button at the top right of the Colab window. Make sure you use the link for your copy in Google Drive (not the link for the original notebook). Then, go to the first text box above, double click on it, and enter your name, student ID number, and the link (URL) to *your copy* of this notebook.

* Upload the PDF file to [Gradescope](https://www.gradescope.com/courses/) in the assignment **P3-PDF**. There are **TBD** analysis questions, so **TBD** questions in Gradescope that you need to align with your PDF.

  * To create the final PDF submission file, make sure to save all cell results you ran.  Make sure that the generated PDF contains all the codes and printed outputs before submission. You are responsible for uploading the correct PDF with all the information required for grading.

  * You can generate a PDF using *File->Print->Save as PDF*.

* Upload the `.ipynb` notebook file to [Gradescope](https://www.gradescope.com/courses/) in the assignment **P3**.
  * You can create this file using *File->Download->Download .ipynb*.


In [ ]:
version = 2.4  # DO NOT MODIFY. If notebook does not match with autograder version, all tests will fail!!

<a name="dataformat"></a>
# 0. Prelude

Here are a list of provided files that will be loaded into your Google Drive for use in this notebook:

- `P3-data-documents.tsv` file contains the collection of documents that we will used in this assignment.
  - The first column is the document name/id (corresponding to the document name/id in the qrels files),
  - The second column is the document content, which has already been preprocessed with tokenizer and stemmer.

- `P3-data-qrels.qrels` file contains relevance judgment information: given a query and a document, is the document relevant to the query? It is another space-separated text file:
  - The first column is the query name/id (corresponding to the query name/id in the trecrun files),
  - The second column is unused (it is present for historical reasons; you won't need to do anything with it except be sure you read it to get to the remaining columns),
  - The third column is a document identifier (“docid”),
  - The fourth column is a number representing the relevance of the document, either 0 for non-relevant, or positive for relevant.
  
- `P3-query-public.tsv` and `P3-query-protected.tsv` files contain the list of query that will be used in public tests and protected test.
  - The first column is the query name/id (corresponding to the document name/id in the qrels files),
  - The second column is the query content, which has already been preprocessed with tokenizer and stemmer.

Similar to P1, **a large number of query-docid pairs will be unjudged, so will not appear in the qrels file. When you encounter that, you should assume that the query-docid pair is non-relevant** (i.e., has a relevance score of zero).  


## 0.1 Expectation & Autograder Tests

Your code will read in the provided input and stopwords file. The correctness of your submission for this assignment will be evaluated using the Gradescope Autograder, which will invoke the following functions to run its tests:
- `tf(inverted_index: InvertedIndex, term: str, doc_id: str) -> int`
- `df(inverted_index: InvertedIndex, term: str) -> int`
- `cf(inverted_index: InvertedIndex, term: str) -> int`
- `tfm(inverted_index: InvertedIndex, queries: dict[str, str], top_k: int) -> dict[str, list[tuple[str, float]]]`
- `bm25(inverted_index: InvertedIndex, queries: dict[str, str], b: float, k1: float, top_k: int) -> dict[str, list[tuple[str, float]]]`
- `ql(inverted_index: InvertedIndex, queries: dict[str, str], lambda_factor: float, top_k: int) -> dict[str, list[tuple[str, float]]]`

<!-- Functions marked with (*) are optional and provide opportunities to earn extra credit. -->

To pass the tests, it is essential to match all type hints and function signatures exactly as specified. The autograder will call these functions using only the parameters defined in the template and will expect outputs in the correct data type. You are free to define additional supporting functions, classes, or use standard Python libraries to implement the specified functions in any way that suits your approach. However, these supplementary implementations should not modify or conflict with the required function signatures or the expected input/output behavior.

Please ensure that **all cells** in your notebook can compile successfully, both locally and on Gradescope. The autograder will verify the notebook's integrity, compatibility, and version upon submission. Failure in any of these checks will result in all tests failing.

There are two kinds of tests for this assigmnet:
* Public Tests: These tests use `P3-query-public.tsv`.
    - Autograder would show the difference between your results and expected results if the tests fail. You can debug using this input.
    - For term statistics related tests, the autograder will only test a subset of `postinglist`, `cf_cache`, and `df_cache`.
* Protected Tests: These tests use `P3-query-protected.tsv`.
    - Autograder **does not** show the debug messages if any of these tests fail.
    - In addition to the query difference, the autograder would also test a different subset of `postinglist`, `cf_cache`, and `df_cache` and tryout different parameters for retrieval models.
* **There is no private test for this assignment.**


## 0.2 Setup

<a name="downloaddata"></a>

We first execute the following to connect to Google Drive (you will be prompted repeatedly for access to your Google Drive; please give it permission) and download copies of the sample files listed above. You should not need to make any modifications to the code, though if you want to use a slightly different path in Google Drive, you can modify the appropriate data_path value. (The autograder will not use your Google Drive.)

In [ ]:
import numpy as np
from tabulate import tabulate
import os

from collections import Counter, defaultdict

try:
    from google.colab import drive

    in_colab = True
except ImportError:
    in_colab = False


# You are more than welcome to code some helper functions.
# But do note that we are only grading functions that are coded in the template files.


# Connect to Google Drive and download copies of the sample files listed above.
# Please allow the access to your Google Drive or the following dataset loader will fail.
# (The autograder will not use your Google Drive.)
if in_colab:
    drive.mount("/content/drive/")  ## DO NOT MODIFY THIS LINE
    data_path = "/content/drive/MyDrive/COMPSCI446/P3"  ## CHANGE TO YOUR OWN FOLDER ON GOOGLE DRIVE
else:
    data_path = "./data/"  ## DO NOT MODIFY THIS LINE. CHANGING THIS LINE WOULD RESULT IN FAIL OF AUTOGRADER TESTS

assert os.path.exists(
    data_path
), "Change data_path to a valid and existing file path in your google drive!"

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


We will now load the texts that stored at `data_path` so that you can work with them in the rest of the notebook. We will not bother loading the files if they have already been loaded to your Google Drive, so this should be a one-time effort.


The code below also loads the files into lists of strings so that you can use them directly elsewhere. That is, you do not need to open and read in the files yourself: it's done.

Note, in this assignment, to practice the whole preprocessing process, we load the entire file directly into memory and then process it line by line. However, in the real-world scenario, when dealing with a large dataset (saying billions of entries), it would be better to process the content line by line to save memory of the machine and/or because the whole file wouldn't even fit in memory.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path


def defaultdict_to_dict(d):
    return (
        {k: defaultdict_to_dict(v) for k, v in d.items()}
        if isinstance(d, defaultdict)
        else d
    )


def download_file(
    file_path: str,
) -> tuple[dict[str, str], dict[str, dict[str, int]], dict[str, str], dict[str, str]]:
    """
    Download necessary from remote. Offload required contents into variables

    Args:
        file_path: the name of file we want to download

    Returns: a tuple of contents loaded from the files.
    """
    webloc = "https://people.cs.umass.edu/~rahimi/446-S25-assignments/P3/"

    data_info = Path(data_path)
    if not data_info.exists() or not data_info.is_dir():
        print(
            f'Google folder "{data_path}" is not present or not a folder. Nothing will work from here.'
        )
        return []

    local_storage_path = Path(os.path.join(data_path, file_path))
    if local_storage_path.is_file():
        print(f'File "{file_path}" already exists, not downloading.')
    else:
        print(f'Cannot find "{file_path}" at "{data_path}" so downloading it')
        urllib.request.urlretrieve(webloc + file_path, local_storage_path)
        print("Done")

    with zipfile.ZipFile(local_storage_path, "r") as zip_ref:
        print(f'Unzipping "{data_path}"')
        zip_ref.extractall(data_path)

    file_names = [
        "P3-data-documents.tsv",
        "P3-data-qrels.qrels",
        "P3-query-public.tsv",
        "P3-query-protected.tsv",
    ]
    contents = []
    for file_name in file_names:
        file_path = os.path.join(data_path, "data", file_name)
        with open(file_path, "r", encoding="utf-8-sig") as f:
            if file_name.endswith(".qrels"):
                qrels_dict = defaultdict(lambda: defaultdict(int))
                for line in f:
                    qid, _, doc_id, relevance = line.strip().split()
                    qrels_dict[str(qid)][str(doc_id)] = int(relevance)
                contents.append(defaultdict_to_dict(qrels_dict))
            elif file_name.endswith(".tsv"):
                tsv_dict = {}
                for line in f:
                    content_id, content_text = line.strip().split("\t")
                    tsv_dict[str(content_id)] = content_text
                contents.append(defaultdict_to_dict(tsv_dict))
            else:
                raise ValueError(f"Unrecognized File Type: {file_name}")
    return contents


# path to p3 data zip file
data_zip_filepath = "P3-data.zip"
documents, qrels, public_queries, protected_queries = download_file(data_zip_filepath)

sample_docs_id = ["4456455", "6816802"]
sample_docs = {
    sample_doc_id: documents[sample_doc_id] for sample_doc_id in sample_docs_id
}

print(
    tabulate(
        list(sample_docs.items()), headers=["Doc ID", "Doc Content"], tablefmt="grid"
    )
)

File "P3-data.zip" already exists, not downloading.
Unzipping "/content/drive/MyDrive/COMPSCI446/P3"
+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|   Doc ID | Doc Content                                                                                                                                                                                                                                                                                                                  |
+==========+===============================================================================================================================================================================================================================

## 0.3 Dataset \& Corresponding Variables

The document, queries, and qrels are selected from the MS MACRO dataset. The data has been tokenized and stemmed while the stopwords are not removed. Note that this means that the tokenization and stemming steps (as in P1) have been performed and, because stemming has already happened, that _you will not use a stopword list_.

The format of the loaded variables are the following:
- `documents`: the variable that contains all documents with document id as key and document text as value. Note that the `doc_text` has already been tokenized and stemmed. To get the terms for each document, you only need to split by space character.
    - ```
    {doc_id: doc_text}
    ```
- `qrels`: the variable that contains the query-docid pair relevance score.
    - ```
    {qid: {doc_id: doc_rel}}
    ```
- `*_queries`: the varaibles that contain the queries which would be used in public and protected tests. Note that the `q_text` has already been tokenized and stemmed. To get the terms for each query, you only need to split by space character.
    - ```
    {qid: q_text}
    ```

<a name="document-parsing"></a>
# 1. Indexing & Term Statistics

In this section, we would focus on building an inverted index for a collection of documents and obtain some term statistics functions which would be used in the later retrieval model section.

We do not evaluate the structure and correctness of the `InvertedIndex` class. However, the instance of this class will be used to evaluate your implementation of term statistics and retrieval models in [Section 1.2](#term-statistics) and [Section 2](#retrieval-models).

## 1.1. Inverted Index

In this section, we implement the inverted index, a fundamental data structure in lexical retrieval models such as BM25 and Query Likelihood (QL). The simplest inverted index object can be constructed by a `dictionary`, where it associates each unique term in the `dictionary` with a list of postings (`posting list`). Each posting in the list captures key term statistics from a document (term frequency in this assignment).

In this assignment, you are expected to implement the `InvertedIndex` class with **attributes** that
1. capture some collection-wise statistics, and
    - `doc_cnt`: The total number of documents in the collection
    - `doc_avg_len`: Average length of documents in the collection
    - `total_term_cnt`: The total number of terms in the collection
    - `doc_ID_len_mapping`: Length of each document in the collection

2. maintain the frequency-related statistics.
    - `df_cache`
        - Store the document frequency for each term, i.e., how many documents contain each term.
        - The format should be `dict[str, int]`, where the key is the term itself and value is the document frequency for the key.
    - `cf_cache`
        - Store the collection frequency for each term, i.e., how many times that the term appears in the entire collection.
        - The format should be `dict[str, int]`, where the key is the term itself and value is the collection frequency for the key.
    - `postings_lists`
        - Store the term-frequency with respect to the docs.
        - For this assignement, the format for `postings_lists` variable should be `dict[str, dict[str, int]]`, e.g.:
        ```python
        {
            term1: {
                doc_id_1: tf_of_term1_in_doc_id_1,
                doc_id_2: ...
                ...
            },
            ...
        }
        ```
        - Note that in the real-world application, the postings are expected also record the location of the term in the document for phrase related operation. Since in this assignment we do not implement those functions or models, we can keep it as an int.
        - Also, though we uses dictionary-based posting list in this assignment,    the number of documents in the real-world collection is often very large, making the idea of storing a dictionary-based posting list not ideal.

In addition, you are expected to implement the **methods** that correctly updates the variables mentioned above and the **functions** that correctly utilize those class methods.
1. Suggested `InvertedIndex` Class Methods:
    - `__init__(self) -> None`
        - The constructor of the class, need to properly initialize the class variables.
    - `add(self, doc_id: str, doc_content: str) -> None`
        - Parse and add the document statistics to the `InvertedIndex` object.
2. Suggested Function:
    - `add_one_document_to_index(inverted_index: "InvertedIndex", doc_id: str, doc_terms: list[str]) -> None`
        - Parse and add document statistics to the `InvertedIndex` object.
        - Note that you can either handle related parsing in here and/or in `InvertedIndex.add()`.
        - We also provide the example that helps to debug your code below.
3. **Required** Function:
    - `build_inverted_index(documents: list[dict[str, str]]) -> InvertedIndex`
        - Parse the documents collection into an `InvertedIndex` class instance.
        - **The autograder will use the object constructed by this function for grading.**

<!-- -
  Keeps the list of unique terms in the collection
  - Keeps some document-independent statistics for each term (the value is not dependent on specific documents):
      - document frequency of terms (the number of documents that have the term)
      - collection frequecny of terms
3.   Postings lists
  - Each term (e.g., `"term_1"`) is associated to a list of postings.
  - Each posting related to document $d$ should include the term's frequency within that document.
-->



<!-- ```python
inverted_index = {
    "term_1": [
        posting_for_doc_1,
        posting_for_doc_2,
        ...
    ],
    "term_2": [
        ...
    ],
    ...
}
``` -->



In [ ]:
from collections import Counter
from math import log

class InvertedIndex:
    doc_ID_len_mapping: dict[str, int]  # mapping from doc_id to its length
    doc_cnt: int  # the number of documents in the collection
    doc_avg_len: float  # average document length
    total_term_cnt: int  # total amount of terms in the collection

    df_cache: dict[str, int]  # stores mapping from term to its document frequency
    cf_cache: dict[str, int]  # stores mapping from term to its collection frequency
    postings_lists: dict[
        str, dict[str, int]
    ]  # records the term frequency statistics for each term and document

    #########
    ##
    ## Implement the class here
    ##
    #########
    def __init__(self):
      self.doc_cnt = 0 # total number of documents
      self.doc_avg_len = None # average document length
      self.total_term_cnt = 0 # total number of terms in the collection

      self.doc_ID_len_mapping = {} # doc_id to length mapping
      self.df_cache = defaultdict(int) # term to document frequency (number of docs the term is in)
      self.cf_cache = defaultdict(int) # term to collection frequency (number of occurrences of the term)

      # term to postings dict
      # postings dict maps doc_id to tf(doc_id)
      # self.postings_lists = defaultdict(dict)
      self.postings_lists = {}

    def add(self, doc_id: str, doc_text: str):
      # for debugging...
      assert doc_id not in self.doc_ID_len_mapping
      # tokenize the document
      doc_tokens = doc_text.strip().split(" ")
      counted_tokens = Counter(doc_tokens)
      doc_len = len(doc_tokens)

      # update stats
      self.doc_cnt += 1
      self.total_term_cnt += doc_len
      self.doc_avg_len = self.total_term_cnt / self.doc_cnt
      self.doc_ID_len_mapping[doc_id] = doc_len
      for term, count in counted_tokens.items():
        self.df_cache[term] += 1
        self.cf_cache[term] += count
        if term not in self.postings_lists:
          self.postings_lists[term] = {}
        self.postings_lists[term][doc_id] = count

    def tf(self, doc_id: str, term: str):
      if term not in self.postings_lists:
        return 0
      if doc_id not in self.postings_lists[term]:
        return 0
      return self.postings_lists[term][doc_id]

    def df(self, term: str):
      return self.df_cache[term]

    def cf(self, term: str):
      return self.cf_cache[term]

    @staticmethod
    def _topk(counter: Counter, top_k: int) -> list[tuple[str, float]]:
      result = counter.most_common()
      return sorted(result, key=lambda tup: (-tup[1], tup[0]), reverse=False)[:top_k]

    def tfm(self, query_text: str, top_k: int) -> list[tuple[str, float]]:
      # little helper function...
      def dict_mult(d: dict[str, int], mult: int) -> dict[str, int]:
        return {k: float(v * mult) for k, v in d.items()}
      # tokenize the query
      query_toks = query_text.strip().split(" ")
      counted_tokens = Counter(query_toks)
      # score each document
      score_counter = Counter()
      for term, count in counted_tokens.items():
        score_counter.update(dict_mult(self.postings_lists.get(term, {}), count))

      return self._topk(score_counter, top_k)

    def bm25(self, query_text: str, b: float, k1: float, top_k: int) -> list[tuple[str, float]]:
      counted_tokens = Counter(query_text.strip().split(' '))

      # get the set of all docs that contain any of the query terms
      rel_docs = set()
      for term in counted_tokens.keys():
        rel_docs.update(self.postings_lists.get(term, {}).keys())

      # for each document, compute the bm25 score
      scores = {k: 0.0 for k in rel_docs}
      for query_term, query_tf in counted_tokens.items():
        for doc_id, doc_tf in self.postings_lists.get(query_term, {}).items():
          doc_len = self.doc_ID_len_mapping[doc_id]
          numer = (k1 + 1) * doc_tf
          denom = k1 * (1 - b + b * (doc_len / self.doc_avg_len)) + doc_tf
          collec = log((self.doc_cnt - self.df(query_term) + 0.5) / (self.df(query_term) + 0.5))
          scores[doc_id] += query_tf * numer * collec / denom

      # sort by score and return the topk
      score_counter = Counter(scores)
      return self._topk(score_counter, top_k)

    def ql(self, query_text: str, lambda_factor: float, top_k: int) -> list[tuple[str, float]]:
      query_toks = query_text.strip().split(' ')
      counted_tokens = Counter(query_toks)

      # get the set of all docs that contain any of the query terms
      rel_docs = set()
      for term in counted_tokens.keys():
        rel_docs.update(self.postings_lists.get(term, {}).keys())

      # iterate through each term in the query
      scores = {k: 0.0 for k in rel_docs}
      for query_term, query_tf in counted_tokens.items():
        # query_mle = query_tf / len(query_toks)
        query_mle = query_tf
        for doc_id in rel_docs:
          doc_tf = self.tf(doc_id, query_term)
          doc_len = self.doc_ID_len_mapping[doc_id]
          doc_mle = doc_tf / doc_len
          collec_mle = self.cf(query_term) / self.total_term_cnt
          logit = (1 - lambda_factor) * doc_mle + lambda_factor * collec_mle
          if logit == 0:
            score = 0
          else:
            score = log(logit)
          scores[doc_id] += query_mle * score

      # sort by score and return the topk
      score_counter = Counter(scores)
      return self._topk(score_counter, top_k)



### 1.1.1 Add Document to `InvertedIndex`

A posting is a data structure that stores term-specific statistics for a given document.
Each unique term will have its own posting list, and for every document, each unique term is associated with one posting.
In this assignment, we simplify the structure of the posting list as a dictionary, mapping the `doc_id` to the term frequency (`tf`) for each term.

The function `add_one_document_to_index()` takes the current inverted index (`InvertedIndex`), along with a document ID (`str`) and its content (`str`), and updates the variables in the `InvertedIndex` class to correctly record the statistics.

  - Note that the document content is provided as a preprocessed string, so just split it by whitespace to extract the document terms.

<font color=red>
**The function and class in the following cell can be fully customized, the autograder will not explicitly test the input and the output of them. It is suggested to use this function when calling `build_inverted_index(documents: list[dict[str, str]])`.**</red>


In [ ]:
def add_one_document_to_index(
    inverted_index: "InvertedIndex", doc_id: str, doc_terms: list[str]
) -> None:
    """
    Updates the inverted index with term frequency from the provided document.

    Args:
        inverted_index: The current inverted index.
        doc_id: The id for the document.
        doc_terms: The list of preprocessed terms in the document doc_id

    Returns:
        No Return as object updates do not require a variable reassignment
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    # hah
    inverted_index.add(doc_id, " ".join(doc_terms))


In [ ]:
# Check the provided sample results with your output!
sample_inverted_index = InvertedIndex()
add_one_document_to_index(
    inverted_index=sample_inverted_index,
    doc_id="4456455",
    doc_terms=documents["4456455"].strip().split(),
)

expected_posting_results = {
    "rang": {"4456455": 1},
    "shown": {"4456455": 1},
    "right": {"4456455": 1},
    "level": {"4456455": 1},
    "100": {"4456455": 1},
    "pokã": {"4456455": 1},
    "mon": {"4456455": 1},
    "maximum": {"4456455": 1},
    "valu": {"4456455": 2},
    "base": {"4456455": 2},
    "benefici": {"4456455": 1},
    "natur": {"4456455": 2},
    "252": {"4456455": 1},
    "ev": {"4456455": 2},
    "31": {"4456455": 1},
    "iv": {"4456455": 2},
    "minimum": {"4456455": 1},
    "hinder": {"4456455": 1},
    "0": {"4456455": 2},
    "type": {"4456455": 2},
    "defens": {"4456455": 1},
    "effect": {"4456455": 1},
    "each": {"4456455": 1},
    "bulbasaur": {"4456455": 1},
    "nor": {"4456455": 1},
}
table_data = [
    [
        term,
        expected_posting_results.get(term, "N/A"),
        dict(sample_inverted_index.postings_lists.get(term, "N/A")),
    ]
    for term in sample_inverted_index.postings_lists.keys()
]
print(
    tabulate(
        table_data, headers=["Term", "Expected Result", "Your Result"], tablefmt="grid"
    )
)

+-----------+-------------------+----------------+
| Term      | Expected Result   | Your Result    |
+===========+===================+================+
| rang      | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| shown     | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| right     | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| level     | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| 100       | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| pokã      | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| mon       | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| maximum   | {'4456455': 1}    | {'4456455': 1} |
+-----------+-------------------+----------------+
| valu      | {'4456455': 2}   

### 1.1.2 Build Inverted Index

Having `add_one_document_to_index()`, the next step is to build the full inverted index by reading and adding every document in the collection.

The function `build_inverted_index()` gets the list of documents in the collection (`list[dict]`) and returns the constructed inverted index object (`InvertedIndex`). For each document in the collection, you will need to:

  - Extract the document's ID and content.
  - Use `add_one_document_to_index()` to update the inverted index with information from this document.

Recall that each document in the collection follows the below format,
```python
{
    "storyID" : "sample-doc-id",
    "text" : "sample-doc-content",
    ... # (some other fields)
}
```




\*\***Autograder will be using the output of this funciton to evaluate your implementation of metrics mentioned in the following section. Make sure the function is outputting an InvertedIndex instance.**

In [ ]:
# Autograder will be using the output of this funciton to evaluate your implementation of metrics mentioned in the following section.
# Make sure the function is outputting an InvertedIndex instance.
def build_inverted_index(documents: dict[str, str]) -> InvertedIndex:
    """
    Constructs an inverted index from a collection of documents.

    Args:
        documents: A list of document dictionaries.

    Returns:
        An inverted index (InvertedIndex Object) containing expected attributes specified in the cell above.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    index = InvertedIndex()
    for doc_id, doc_text in documents.items():
      index.add(doc_id, doc_text)
    return index

    return InvertedIndex()  # You will return something meaningful before this statement

In [ ]:
# Check the provided sample results with your output!
sample_inverted_index = build_inverted_index(documents=sample_docs)

expected_postings_results = {
    "rang": {"4456455": 1},
    "shown": {"4456455": 1},
    "right": {"4456455": 1},
    "level": {"4456455": 1},
    "100": {"4456455": 1},
    "pokã": {"4456455": 1},
    "mon": {"4456455": 1},
    "maximum": {"4456455": 1},
    "valu": {"4456455": 2},
    "base": {"4456455": 2, "6816802": 1},
    "benefici": {"4456455": 1},
    "natur": {"4456455": 2},
    "252": {"4456455": 1},
    "ev": {"4456455": 2},
    "31": {"4456455": 1},
    "iv": {"4456455": 2},
    "minimum": {"4456455": 1},
    "hinder": {"4456455": 1},
    "0": {"4456455": 2},
    "type": {"4456455": 2, "6816802": 1},
    "defens": {"4456455": 1},
    "effect": {"4456455": 1},
    "each": {"4456455": 1},
    "bulbasaur": {"4456455": 1},
    "nor": {"4456455": 1},
    "energi": {"6816802": 1},
    "store": {"6816802": 1},
    "electr": {"6816802": 3},
    "vehicl": {"6816802": 3},
    "recharg": {"6816802": 2},
    "batteri": {"6816802": 2},
    "suppli": {"6816802": 2},
    "power": {"6816802": 2},
    "motor": {"6816802": 4},
    "control": {"6816802": 3},
    "devic": {"6816802": 1},
    "which": {"6816802": 1},
    "amount": {"6816802": 1},
    "drive": {"6816802": 1},
    "s": {"6816802": 1},
    "posit": {"6816802": 1},
    "acceler": {"6816802": 1},
    "automobil": {"6816802": 1},
    "truck": {"6816802": 1},
    "bu": {"6816802": 1},
    "use": {"6816802": 1},
    "fuel": {"6816802": 2},
    "replac": {"6816802": 1},
    "gasolin": {"6816802": 1},
    "diesel": {"6816802": 1},
    "other": {"6816802": 1},
    "combust": {"6816802": 2},
    "gone": {"6816802": 1},
    "intern": {"6816802": 1},
    "engin": {"6816802": 1},
    "transmiss": {"6816802": 1},
}
table_data = [
    [
        term,
        expected_postings_results.get(term, "N/A"),
        dict(sample_inverted_index.postings_lists.get(term, "N/A")),
    ]
    for term in sample_inverted_index.postings_lists.keys()
]
print(
    tabulate(
        table_data, headers=["Term", "Expected Result", "Your Result"], tablefmt="grid"
    )
)

+-----------+------------------------------+------------------------------+
| Term      | Expected Result              | Your Result                  |
+===========+==============================+==============================+
| rang      | {'4456455': 1}               | {'4456455': 1}               |
+-----------+------------------------------+------------------------------+
| shown     | {'4456455': 1}               | {'4456455': 1}               |
+-----------+------------------------------+------------------------------+
| right     | {'4456455': 1}               | {'4456455': 1}               |
+-----------+------------------------------+------------------------------+
| level     | {'4456455': 1}               | {'4456455': 1}               |
+-----------+------------------------------+------------------------------+
| 100       | {'4456455': 1}               | {'4456455': 1}               |
+-----------+------------------------------+------------------------------+
| pokã      

<a name="term-statistics"></a>
## 1.2. Term Statistics ( Points)

The inverted index does more than just storing term occurrences: it provides the basis for computing key term statistics like term frequency (`TF`), document frequency (`DF`), and collection frequency (`CF`). These statistics are essential for retrieval models such as Best Match 25 (`BM25`) and Query Likelihood (`QL`). In this section, we'll use the inverted index to calculate these term statistics before moving on to the retrieval models.

### 1.2.1 Term Frequency ( Points)

First, we want to get the frequency of term $w$ in document $d$, using the built inverted index.

`tf(inverted_index: InvertedIndex, term: str, doc_id: str) -> int`

Note that in efficient implementation of retrieval models, we don't use this function.

In [ ]:
def tf(inverted_index: InvertedIndex, term: str, doc_id: str) -> int:
    """
    Get the term frequency (TF) for a given term in a specific document.

    Args:
        inverted_index (InvertedIndex): The inverted index data structure.
        term (str): The term for which we want to get the frequency.
        doc_id (str): The document id in which to count the term's occurrences.

    Returns:
        int: The frequnecy of the term in the document doc_id.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    return inverted_index.tf(doc_id, term)

    return 0  # You will return something meaningful before this statement


# Check the provided sample results with your output!
sample_term = "base"
sample_doc_id = sample_docs_id[0]
sample_tf = tf(
    inverted_index=sample_inverted_index, term=sample_term, doc_id=sample_doc_id
)

expected_sample_tf = 2
print(f"Expected Sample Outputs: {expected_sample_tf}")
print(f"  Your   Sample Outputs: {sample_tf}")

Expected Sample Outputs: 2
  Your   Sample Outputs: 2


### 1.2.2 Document Frequency ( Points)

Here, we want to compute the number of documents in the collection $C$ that contains the term $t$ using the inverted index.

`df(inverted_index: InvertedIndex, term: str) -> int`



In [ ]:
def df(inverted_index: InvertedIndex, term: str) -> int:
    """
    Calculate the document frequency (DF) for a given term in a collection C.

    Args:
        inverted_index (InvertedIndex): The inverted index data structure
        term (str): The term

    Returns:
        int: The number of documents in the collection $C$ that contains the term.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    return inverted_index.df(term)

    return 0  # You will return something meaningful before this statement


# Check the provided sample results with your output!
sample_term = "base"
sample_df = df(inverted_index=sample_inverted_index, term=sample_term)

expected_sample_df = 2
print(f"Expected Sample Outputs: {expected_sample_df}")
print(f"  Your   Sample Outputs: {sample_df}")

Expected Sample Outputs: 2
  Your   Sample Outputs: 2


### 1.2.2 Collection Frequency ( Points)

Here, we want to compute the number of occurence for the term $t$ in a collection $C$ using the inverted index.

`cf(inverted_index: InvertedIndex, term: str) -> int`



In [ ]:
def cf(inverted_index: InvertedIndex, term: str) -> int:
    """
    Calculate the collection frequency (CF) of a given term in a collection C.

    Args:
        inverted_index (InvertedIndex): The inverted index
        term (str): The term t

    Returns:
        int: The number of times term t appears in the collection $C$
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    return inverted_index.cf(term)


    return 0  # You will return something meaningful before this statement


# Check the provided sample results with your output!
sample_term = "base"
sample_cf = cf(inverted_index=sample_inverted_index, term=sample_term)

expected_sample_cf = 3
print(f"Expected Sample Outputs: {expected_sample_cf}")
print(f"  Your   Sample Outputs: {sample_cf}")

Expected Sample Outputs: 3
  Your   Sample Outputs: 3


<a name="retrieval-models"></a>
# 2. Retrieval Models ( Points)

In this section, you will implement a simple retrieval model based on TF, as well as the popular BM25 and Query Likelihood (QL) models.

**Note: All logarithms used in the formulas refer to natural logarithms.**

**The output from the each Retrieval Model should follow the format `dict[str, list[tuple[str, float]]]`, e.g.,**
```python
{
    query_id_1: [
        (doc_id_1, doc_id_1_score),
        (doc_id_2, doc_id_2_score),
        ...
        (doc_id_top_k, doc_id_top_k_score),
    ],
    query_id_2: [
        ...
    ],
    ...
}
```
If the number of document in the result ranklist is less than `top_k`, only return the result ranklist and do not add other document to fillup the ranklist. If the length of ranklist is `0`, return an empty list.

## 2.1 Term Frequency Retrieval Model (TFM) ( points)

In the cell below, implement the TF retrieval model using the formula presented below.

\begin{align}
    \mathrm{TF}(q,d)
    = \sum_{t \in {d \ \cap \ q}}
        \mathrm{tf}(t,d)
    \enspace .
\end{align}

- $\mathrm{tf}(t, d)$: the frequency of term $t$ in document $d$.

**It is important to note that all computations are performed at the term level, meaning that a term may appear multiple times in a document or query, and each occurrence is taken into account.**



For each query, you need to return the top 50 documents ranked based on their TF scores.


In [ ]:
def tfm(
    inverted_index: InvertedIndex,
    queries: dict[str, str],
    top_k: int,
) -> dict[str, list[tuple[str, float]]]:
    """
    Compute TF scores for a collection of queries using the provided inverted index.

    Args:
        inverted_index (InvertedIndex): Inverted index with term postings and document lengths.
        queries (dict[str, str]): Mapping from query IDs to query strings.
        top_k (int): Number of top-ranked documents to return per query.

    Returns:
        dict[str, list[tuple[str, float]]]: Mapping from query IDs to lists of (doc_id, score) tuples,
                                             sorted by descending TF score.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    results = {}
    for q_id, q_text in queries.items():
      results[q_id] = inverted_index.tfm(q_text, top_k)
    return results

    return {
        "sample_query_id": []
    }  # You will return something meaningful before this statement

In [ ]:
# Check the provided sample results with your output!
top_k = 50
sample_tfm_result = tfm(
    inverted_index=sample_inverted_index, queries=public_queries, top_k=top_k
)

expected_sample_tfm_result = {
    "156493": [],
    "1110199": [],
    "1063750": [],
    "130510": [],
    "489204": [("4456455", 1.0)],
    "573724": [],
    "168216": [],
    "1133167": [],
    "527433": [("4456455", 2.0), ("6816802", 1.0)],
    "1037798": [],
    "915593": [("4456455", 2.0), ("6816802", 1.0)],
    "264014": [],
    "1121402": [],
    "962179": [],
    "1117099": [],
    "148538": [],
    "451602": [],
    "359349": [],
    "1115776": [],
    "1112341": [],
}
table_data = [
    [qid, expected_sample_tfm_result.get(qid, "N/A"), sample_tfm_result.get(qid, "N/A")]
    for qid in public_queries.keys()
]
print(
    tabulate(
        table_data,
        headers=["Query ID", "Expected Ranklist", "Your Ranklist"],
        tablefmt="grid",
    )
)

+------------+--------------------------------------+--------------------------------------+
|   Query ID | Expected Ranklist                    | Your Ranklist                        |
+============+======================================+======================================+
|     156493 | []                                   | []                                   |
+------------+--------------------------------------+--------------------------------------+
|    1110199 | []                                   | []                                   |
+------------+--------------------------------------+--------------------------------------+
|    1063750 | []                                   | []                                   |
+------------+--------------------------------------+--------------------------------------+
|     130510 | []                                   | []                                   |
+------------+--------------------------------------+-----------------

## 2.2 Best Match 25 (BM25) ( points)

In the cell below, implement the BM25 retrieval model using the formula presented below.

\begin{align}
    \mathrm{BM25}(q,d)
    = \sum_{t \in {d \ \cap \ q}}
        \frac{
            (k_1+1) \mathrm{tf}(t,d)
        }{
            k_1( 1-b + b(\frac{|d|}{\mathrm{avgdl}}))
            + \mathrm{tf}(t,d)
        }
        \ln \frac{|C|-\mathrm{df}(t, C)+0.5}{\mathrm{df}(t, C)+0.5}
    \enspace .
\end{align}

- $\mathrm{tf}(t, d)$: the frequency of term $t$ in document $d$.
- $\mathrm{df}(t, C)$: the number of documents containing term $t$ in the collection $C$.
- $|d|$: the length of document $d$ (the total number of terms in $d$).
- $|C|$: the total number of documents in the collection $C$.
- $\mathrm{avgdl}$: the  average length of documents in the collection.
- $b$, $k_1$: the hyper-parameters of the BM25 model. In this project, we are using $b = 0.75, k_1 = 1.2$.

**It is important to note that all computations are performed at the term level, meaning that a term may appear multiple times in a document or query, and each occurrence is taken into account.**


For each query, you need to return the top 50 documents ranked based on their BM25 scores.


In [ ]:
from math import log


def bm25(
    inverted_index: InvertedIndex,
    queries: dict[str, str],
    b: float,
    k1: float,
    top_k: int,
) -> dict[str, list[tuple[str, float]]]:
    """
    Compute BM25 scores for a collection of queries using the provided inverted index.

    Args:
        inverted_index (InvertedIndex): Inverted index with term postings and document lengths.
        queries (dict[str, str]): Mapping from query IDs to query strings.
        b (float): BM25 document length normalization parameter.
        k1 (float): BM25 term frequency scaling parameter.
        top_k (int): Number of top-ranked documents to return per query.

    Returns:
        dict[str, list[tuple[str, float]]]: Mapping from query IDs to lists of (doc_id, score) tuples,
                                             sorted by descending BM25 score.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    result = {}
    for q_id, q_text in queries.items():
      result[q_id] = inverted_index.bm25(q_text, b, k1, top_k)
    return result

    return {
        "sample_query_id": []
    }  # You will return something meaningful before this statement

In [ ]:
# Check the provided sample results with your output!
b = 0.75
k1 = 1.2
top_k = 50
sample_bm25_result = bm25(
    inverted_index=sample_inverted_index,
    queries=public_queries,
    b=b,
    k1=k1,
    top_k=top_k,
)

expected_sample_bm25_result = {
    "156493": [],
    "1110199": [],
    "1063750": [],
    "130510": [],
    "489204": [("4456455", 0.0)],
    "573724": [],
    "168216": [],
    "1133167": [],
    "527433": [("6816802", -1.4877157173760593), ("4456455", -2.3448764287119346)],
    "1037798": [],
    "915593": [("6816802", -1.4877157173760593), ("4456455", -2.3448764287119346)],
    "264014": [],
    "1121402": [],
    "962179": [],
    "1117099": [],
    "148538": [],
    "451602": [],
    "359349": [],
    "1115776": [],
    "1112341": [],
}
table_data = [
    [
        qid,
        expected_sample_bm25_result.get(qid, "N/A"),
        sample_bm25_result.get(qid, "N/A"),
    ]
    for qid in public_queries.keys()
]
print(
    tabulate(
        table_data,
        headers=["Query ID", "Expected Ranklist", "Your Ranklist"],
        tablefmt="grid",
    )
)

+------------+----------------------------------------------------------------------+----------------------------------------------------------------------+
|   Query ID | Expected Ranklist                                                    | Your Ranklist                                                        |
+============+======================================================================+======================================================================+
|     156493 | []                                                                   | []                                                                   |
+------------+----------------------------------------------------------------------+----------------------------------------------------------------------+
|    1110199 | []                                                                   | []                                                                   |
+------------+--------------------------------------------

# 2.4: Query Likelihood (QL) ( points)

In the cell below, implement the Query Likelihood (QL) retrieval model using the Jelinek-Mercer (JM) Smoothing technique.

\begin{align}
    &\mathrm{QL}(q,d)
        = \sum_{t \ \in \ {q}}
        \ln \left( (1-\lambda) P_{\mathrm{MLE}}(t|d) + \lambda P_{\mathrm{MLE}}(t|C) \right)
        \enspace , \enspace
        \text{where}
        \\
    &P_{\mathrm{MLE}}(t|d)
        = \frac{\mathrm{tf}(t,d)}{|d|}
        \enspace ,
        \enspace
    P_{\mathrm{MLE}}(t|C) = \frac{\mathrm{cf}(t,C)}{|C|}
        \enspace .
\end{align}
- $\mathrm{tf}(t, d)$: the frequency of term $t$ in document $d$.
- $\mathrm{cf}(t, C)$: the frequency of term $t$ in the collection $C$.
- $|d|$: the length of document $d$ (the total number of terms in $d$).
- $|C|$: the total number of terms in collection $C$.
- $\lambda$: the hyper-parameters for the QL model. In this project, we are using $\lambda = 0.2$.


**It is important to note that all computations are performed at the term level, meaning that a term may appear multiple times in a document or query, and each occurrence is taken into account.**

Similar to the previous model, return the top 50 retrieved documents for each query ranked based on the QL scores.

In [ ]:
def ql(
    inverted_index: InvertedIndex,
    queries: dict[str, str],
    lambda_factor: float,
    top_k: int,
) -> dict[str, list[tuple[str, float]]]:
    """
    Compute Query Likelihood (QL) scores for queries using the inverted index.

    Args:
        inverted_index (InvertedIndex): Inverted index containing term postings and document lengths.
        queries (dict[str, str]): Mapping from query IDs to query strings.
        lambda_factor (float): Smoothing parameter.
        top_k (int): Number of top-ranked documents to return per query.

    Returns:
        dict[str, list[tuple[str, float]]]: Mapping from query IDs to lists of (doc_id, score) tuples,
                                             sorted by descending QL score.
    """

    #########
    ##
    ## Implement the function here
    ##
    #########
    result = {}
    for q_id, q_text in queries.items():
      result[q_id] = inverted_index.ql(q_text, lambda_factor, top_k)
    return result

    return {
        "sample_query_id": []
    }  # You will return something meaningful before this statement

In [ ]:
# Check the provided sample results with your output!
lambda_factor = 0.2
top_k = 50
sample_ql_result = ql(
    inverted_index=sample_inverted_index,
    queries=public_queries,
    lambda_factor=lambda_factor,
    top_k=top_k,
)

expected_sample_ql_result = {
    "156493": [],
    "1110199": [],
    "1063750": [],
    "130510": [],
    "489204": [("4456455", -3.5935692743096115)],
    "573724": [],
    "168216": [],
    "1133167": [],
    "527433": [("4456455", -2.855970331178832), ("6816802", -3.7227810057896176)],
    "1037798": [],
    "915593": [("4456455", -2.855970331178832), ("6816802", -3.7227810057896176)],
    "264014": [],
    "1121402": [],
    "962179": [],
    "1117099": [],
    "148538": [],
    "451602": [],
    "359349": [],
    "1115776": [],
    "1112341": [],
}
table_data = [
    [qid, expected_sample_ql_result.get(qid, "N/A"), sample_ql_result.get(qid, "N/A")]
    for qid in public_queries.keys()
]
print(
    tabulate(
        table_data,
        headers=["Query ID", "Expected Ranklist", "Your Ranklist"],
        tablefmt="grid",
    )
)

+------------+---------------------------------------------------------------------+---------------------------------------------------------------------+
|   Query ID | Expected Ranklist                                                   | Your Ranklist                                                       |
+============+=====================================================================+=====================================================================+
|     156493 | []                                                                  | []                                                                  |
+------------+---------------------------------------------------------------------+---------------------------------------------------------------------+
|    1110199 | []                                                                  | []                                                                  |
+------------+--------------------------------------------------------

# 3. Analysis Questions ( Points)

## <font color="red"> add eval functions from P2 so that they compare the performance of different models </font>

## 3.1 Answer the following questions.

Note that for this project, "short" and "long" are measured by the number of terms, not the number of characters.

- What is the average length of a story in the *sciam* collection?
- What is the shortest story (and how short it is)?
- What is the longest story (and how long is it)?
- What word occurs in the most stories and how many stories does it occur in?
- What word has the largest number of occurrences and how many does it have?
- How many unique words are there in this collection?
- How many of them occur only once and what percent is that? Is that what you would expect? Why or why not?



**Enter your answer here**

## 3.2
Your training queries have two queries that are roughly about the scientific american supplement. Suppose that you wanted to judge stories for relevance using a pooling strategy that takes the top 100 documents from each of those two queries. How many unique documents will you be judging? What if you only considered the top 20? Suppose you had a budget that allowed you to judge at most 30 documents. How deeply could you go into the two queries for judging to get 30 judged, no more, no less?

**Enter your answer here**

# 4. Misc & Grading

Your P3 submission is graded out of 100 points, allocated as follows:
* 82(+6) points for the code (these are autograded)
  * 72 points for correct evaluation results using different metrics on trecrun of QL, BM25, and DPR runs. (2.1-2.7)
  * Up to 6 points **extra credit** for evaluation metric P@%R and P@R. (2.8,2.9)
  * 10 points for correct implementation on aggregating per query result into per trecrun results. (3.2)

* 18(+4) points for the written analysis questions (these are graded manually):
  * 9 points for interpretation of generated table (4.1.2)
  * 9 points for correctness of recall/precision graph (4.2)
  * Up to 4 points for extra credit interpolation graph (4.3)

Note that we expect that you will upload your submission in the correct format (notebook and PDF), that the code in the notebook will run, and that the code will successfully process provided trecrun files, including some you do not have access to.